# Detective R-GCN: Villain Prediction + Link Prediction

Multi-task R-GCN trained on 576 murder mystery plot graphs.

**Tasks:**
1. **Node classification** — predict Villain / Victim / Witness / Uninvolved
2. **Link prediction** — predict missing edges between characters, locations, etc.

**Reference:** Schlichtkrull et al., *Modeling Relational Data with Graph Convolutional Networks*, ESWC 2018

In [ ]:
import os, pickle, time, torch
import numpy as np

from rgcn_model import (
    DEVICE,
    RGCNMultiTask,
    train_multitask,
    evaluate,
    evaluate_classification,
    plot_multitask_history,
)
from load_mystery_graphs import load_mystery_graphs

print(f"Device: {DEVICE}")

## 1. Configuration

In [ ]:
# Paths
PROJECT_DIR    = os.path.dirname(os.path.abspath("__file__"))
JSON_DIR       = os.path.join(PROJECT_DIR, "extraction", "data", "graphs")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

GRAPH_CACHE = os.path.join(CHECKPOINT_DIR, "detective_graph.pkl")
MODEL_CKPT  = os.path.join(CHECKPOINT_DIR, "rgcn_multitask_detective.pt")

# Hyperparameters
HIDDEN_DIM  = 64
NUM_LAYERS  = 2
NUM_BASES   = 8       # 8 base relations → 8 bases is a natural choice
DROPOUT     = 0.2
LR          = 1e-3
NUM_EPOCHS  = 50
BATCH_SIZE  = 4096
NEG_RATIO   = 3       # more negatives help rare relations (kills, harms)
EVAL_EVERY  = 10
CLS_WEIGHT  = 1.0     # classification loss weight
LINK_WEIGHT = 1.0     # link prediction loss weight

print(f"JSON dir   : {JSON_DIR}")
print(f"Checkpoints: {CHECKPOINT_DIR}")

## 2. Load Data

In [ ]:
if os.path.exists(GRAPH_CACHE):
    print(f"Loading cached graph from {GRAPH_CACHE}")
    with open(GRAPH_CACHE, "rb") as f:
        det_graph = pickle.load(f)
else:
    print("Parsing graph JSONs...")
    det_graph = load_mystery_graphs(JSON_DIR)
    with open(GRAPH_CACHE, "wb") as f:
        pickle.dump(det_graph, f)
    print(f"Graph cached to {GRAPH_CACHE}")

det_graph.summary()

## 3. Class Distribution

In [ ]:
import matplotlib.pyplot as plt

# Show class distribution across splits
for split_name, mask in [("Train", det_graph.train_mask),
                          ("Val",   det_graph.val_mask),
                          ("Test",  det_graph.test_mask)]:
    labels = det_graph.class_labels[mask]
    counts = [(labels == i).sum().item() for i in range(det_graph.num_classes)]
    names  = [det_graph.class_names[i] for i in range(det_graph.num_classes)]
    total  = sum(counts)
    dist   = ", ".join(f"{n}: {c} ({100*c/total:.1f}%)" for n, c in zip(names, counts))
    print(f"{split_name:5s} ({total:,}): {dist}")

## 4. Build Model

In [ ]:
model = RGCNMultiTask(
    num_nodes     = det_graph.num_nodes,
    num_relations = det_graph.num_relations,
    num_classes   = det_graph.num_classes,
    hidden_dim    = HIDDEN_DIM,
    num_layers    = NUM_LAYERS,
    num_bases     = NUM_BASES,
    feat_dim      = det_graph.node_features.shape[1],
    dropout       = DROPOUT,
)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Train or Load

In [ ]:
if os.path.exists(MODEL_CKPT):
    print(f"Checkpoint found — loading from {MODEL_CKPT}")
    model.load_state_dict(torch.load(MODEL_CKPT, map_location=DEVICE))
    model.to(DEVICE).eval()
    history = None
    print("Model loaded (training skipped).")
else:
    print("No checkpoint — training from scratch...")
    history = train_multitask(
        model, det_graph,
        num_epochs  = NUM_EPOCHS,
        lr          = LR,
        batch_size  = BATCH_SIZE,
        eval_every  = EVAL_EVERY,
        neg_ratio   = NEG_RATIO,
        cls_weight  = CLS_WEIGHT,
        link_weight = LINK_WEIGHT,
    )
    torch.save(model.state_dict(), MODEL_CKPT)
    print(f"Model saved to {MODEL_CKPT}")

## 6. Training Curves

In [ ]:
if history is not None:
    plot_multitask_history(history, "Detective R-GCN — Multi-task Training")
else:
    print("Loaded from checkpoint — no training history.")

## 7. Test Set Evaluation

In [ ]:
# Move data to device
if det_graph.node_features is not None:
    det_graph.node_features = det_graph.node_features.to(DEVICE)

train_e    = det_graph.train_edges.to(DEVICE)
edge_index = train_e[:, :2].t().contiguous()
edge_type  = train_e[:, 2]

# Link prediction metrics
lp_metrics = evaluate(model, det_graph, det_graph.test_edges, edge_index, edge_type)
print("Link Prediction (test):")
for k, v in lp_metrics.items():
    print(f"  {k:>8}: {v:.4f}")

# Node classification metrics
test_mask = det_graph.test_mask.to(DEVICE)
test_acc = evaluate_classification(model, det_graph, test_mask, edge_index, edge_type)
print(f"\nNode Classification (test):")
print(f"  Accuracy: {test_acc:.4f}")

## 8. Per-Class Breakdown

In [ ]:
from collections import Counter

model.eval()
with torch.no_grad():
    node_emb = model.encode(
        edge_index, edge_type,
        node_features=det_graph.node_features,
        num_nodes=det_graph.num_nodes,
    )
    logits = model.node_classifier(node_emb)
    preds  = logits[test_mask].argmax(dim=1).cpu()
    truths = det_graph.class_labels[det_graph.test_mask].cpu()

print(f"{'Class':<15} {'Correct':>8} {'Total':>8} {'Acc':>8}")
print("-" * 42)
for cls_id in range(det_graph.num_classes):
    cls_mask = (truths == cls_id)
    total    = cls_mask.sum().item()
    correct  = ((preds == cls_id) & cls_mask).sum().item()
    acc      = correct / total if total > 0 else 0.0
    print(f"{det_graph.class_names[cls_id]:<15} {correct:>8} {total:>8} {acc:>8.3f}")

# Confusion matrix
print("\nConfusion Matrix (rows=true, cols=predicted):")
class_names = [det_graph.class_names[i] for i in range(det_graph.num_classes)]
header = f"{'':>15}" + "".join(f"{n:>12}" for n in class_names)
print(header)
for true_id in range(det_graph.num_classes):
    row_mask = (truths == true_id)
    row_preds = preds[row_mask]
    counts = Counter(row_preds.tolist())
    row = f"{class_names[true_id]:>15}"
    for pred_id in range(det_graph.num_classes):
        row += f"{counts.get(pred_id, 0):>12}"
    print(row)

## 9. Manual Controls

Uncomment cells below to force re-training or rebuild the graph cache.

In [ ]:
# Force rebuild graph cache:
# os.remove(GRAPH_CACHE)
# det_graph = load_mystery_graphs(JSON_DIR)
# with open(GRAPH_CACHE, "wb") as f:
#     pickle.dump(det_graph, f)

# Force retrain:
# os.remove(MODEL_CKPT)